# Rebuild the data: generation, steering, judging

This notebook re-creates everything in `data/` from scratch, in the order it was originally made. It needs:

- a Gemini API key (`GEMINI_API_KEY`) for generation and judging. The models used, `gemini-3.5-flash` and
  `gemini-3.1-flash-lite`, may no longer be offered; the shipped generations and judge outputs are the record
  of those steps, and everything downstream of them (training, steering, the 32B models) needs no Gemini;
- a SLURM cluster with GPUs for AxBench training and steering of Gemma-2-2B-it (Hugging Face access to the
  gated `google/gemma-2-2b-it`), and AxBench at commit `41c8332` with `pipeline/axbench/axbench.patch`;
- one or more 80 GB GPUs for the OLMo-3.1-32B and Qwen3-32B extraction and steering.

Generation is stochastic (Gemini at temperature 1.0 / 0.7), so a rebuild gives new training text and slightly
different vectors, not the shipped ones. To reproduce the paper's numbers exactly, use the shipped data and
`reproduce_from_data.ipynb`.

Every command is printed. **Nothing runs and no job is submitted while `DRY_RUN = True`.** Set the SLURM
partition / account options for your cluster in `SBATCH_ARGS`.

In [ ]:
DRY_RUN = True
SBATCH_ARGS = []        # e.g. ["--partition=gpu", "--account=myproject"]

import os, re, shlex, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (ROOT / "relgeom").is_dir()
os.chdir(ROOT)
PY = sys.executable

TEMP = [f"temp_c1{s}" for s in "abcd"] + [f"temp_f1{s}" for s in "abcdef"] + ["h2o_v1"]
ANALOGY = ["size_imp_v1", "war_arg_v1", "food_ideas_v1", "frames_v1"]
LAYERS = {**{d: [0, 1, 2, 3, 4, 5, 10, 15, 20] for d in TEMP}, **{d: [5, 10, 15, 20] for d in ANALOGY}}
_fake = iter(range(1000, 10**6))


def sh(cmd):
    """Print a shell command; run it unless DRY_RUN."""
    print("$", cmd)
    if not DRY_RUN:
        subprocess.run(cmd, shell=True, check=True, cwd=ROOT)


def sbatch(script, env, after=None):
    """Submit pipeline/slurm/<script> with env vars; returns the job id (a placeholder under DRY_RUN)."""
    exports = ",".join(["ALL"] + [f"{k}={v}" for k, v in env.items()])
    dep = [f"--dependency=afterok:{after}"] if after else []
    cmd = ["sbatch", "--parsable", f"--export={exports}"] + SBATCH_ARGS + dep + [f"pipeline/slurm/{script}"]
    print("$", " ".join(shlex.quote(x) for x in cmd))
    if DRY_RUN:
        return str(next(_fake))
    return subprocess.run(cmd, check=True, capture_output=True, text=True, cwd=ROOT).stdout.strip().split(";")[0]

## 1. Generate training text (Gemini)

The Fahrenheit ladder reuses the Celsius prompt pool, so it runs after the Celsius generator and makes no API
calls. Each generator writes `data/inputs/<ds>_vectors.jsonl` and `data/raw/<ds>/`.

In [ ]:
sh(f"{PY} pipeline/generate/generate_temp_celsius.py")
sh(f"{PY} pipeline/generate/generate_temp_fahrenheit.py")
sh(f"{PY} pipeline/generate/generate_water_words.py")
sh(f"{PY} pipeline/generate/generate_analogies.py --concurrency 48")

## 2. Convert to AxBench format

In [ ]:
for ds in TEMP + ANALOGY:
    sh(f"{PY} pipeline/axbench/convert_to_axbench.py --vectors data/inputs/{ds}_vectors.jsonl "
       f"--raw-dir data/raw/{ds} --out-dir data/axbench/{ds}")

## 3. AxBench

Clone upstream AxBench at the commit the paper used, apply the patch (paired negatives, held-out prompts from
`heldout_eval.parquet`, truncation logging, lazy LM loading), and build its environment. AxBench's `train.py`
imports `stanza`, which its lock file does not list, so install it explicitly. Then write one config per dataset
and layer from `pipeline/axbench/config_template.yaml`.

In [ ]:
sh("git clone https://github.com/stanfordnlp/axbench ax_bench/axbench && cd ax_bench/axbench && git checkout 41c8332 "
   "&& git apply ../../pipeline/axbench/axbench.patch")
sh("cd ax_bench/axbench && uv sync && uv pip install stanza")
sh(f"LAYERS='0 1 2 3 4 5 10 15 20' {PY} pipeline/axbench/make_configs.py " + " ".join(TEMP))
sh(f"{PY} pipeline/axbench/make_configs.py " + " ".join(ANALOGY))

## 4. Train, calibrate, steer and judge (SLURM)

Per dataset and layer, four jobs chained with `afterok`: DiffMean training (`ds_train`), `max_act` calibration
(`ds_latent`, `pipeline/axbench/compute_max_act.py`), steering on the first 12 held-out prompts at 15 factors
(`ds_infer`), and the Gemini judge (`ds_eval_gemini`; needs outbound internet).

In [ ]:
jobs = {}
for ds, layers in LAYERS.items():
    for L in layers:
        env = dict(DATASET=ds, CONFIG_PREFIX=f"diffmean_{ds}_g2-2b", LAYER=L)
        j = sbatch("ds_train.slurm", env)
        j = sbatch("ds_latent.slurm", env, after=j)
        j = sbatch("ds_infer.slurm", env, after=j)
        jobs[ds, L] = sbatch("ds_eval_gemini.slurm", env, after=j)
print(len(jobs), "chains")

When every chain has finished, merge the temperature chunks into `temp_c1` and `temp_f1` (vectors, concept
index and judge scores, temperature-ordered).

In [ ]:
ALL = "0 1 2 3 4 5 10 15 20"
sh(f"{PY} pipeline/axbench/recombine_temp_c1.py --layers {ALL}")
sh(f"{PY} pipeline/axbench/recombine_temp_f1.py --layers {ALL}")

## 5. OLMo-3.1-32B and Qwen3-32B (GPU)

DiffMean vectors at every layer from the same training text (`extract.py`), then steering at the depth-matched
layers on the reduced grid (`run_steer.sh`: 9 factors, first 6 prompts), the judge, and the chunk merge. Only
the nine depth-matched layers are shipped in `data/models/`.

In [ ]:
for tag in ["olmo31_32b", "qwen3_32b"]:
    sh(f"{PY} pipeline/bigmodel/extract.py --tag {tag} --datasets all")
sh("bash pipeline/bigmodel/run_steer.sh")
for tag in ["olmo31_32b", "qwen3_32b"]:
    for ds in TEMP + ANALOGY:
        for L in [12, 25, 37]:
            d = f"data/models/{tag}/axbench/{ds}/diffmean_l{L}"
            sh(f"[ -f {d}/inference/steering_data.parquet ] && {PY} pipeline/judge/gemini_judge.py "
               f"--steering-data {d}/inference/steering_data.parquet --concept-index data/axbench/{ds}/concept_index.parquet "
               f"--out-dir {d}/eval --model gemini-3.1-flash-lite --concurrency 48 || true")
    for u in ["c1", "f1"]:
        sh(f"{PY} pipeline/axbench/recombine_temp_{u}.py --base data/models/{tag}/axbench --layers {' '.join(map(str, range(64)))}")

## 6. GemmaScope activations (GPU), for the SAE analysis

In [ ]:
sbatch("sae_extract.slurm", {})

## 7. Analyses

With the data rebuilt, run `reproduce_from_data.ipynb`.